# 03. 回収率（ROI）分析

モデル予測確率・各種条件での単勝回収率を評価し、プラス期待値の買い目を探索する。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys; sys.path.append('../src')
from evaluate_roi import roi_by_bet_condition, roi_from_model_proba, kelly_bet_sizes, roi_by_factor

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_theme(style='whitegrid')

df = pd.read_csv('../data/processed/features_with_pred.csv', parse_dates=['race_date'])
# OOFが存在する行のみ（最初のfoldはOOF=0になる可能性があるため除外）
df = df[df['pred_win_proba'] > 0].copy()
print(df.shape)

## 1. 予測確率閾値 vs 回収率

In [ ]:
thresholds = np.arange(0.05, 0.55, 0.025)
results = []
for thr in thresholds:
    r = roi_from_model_proba(df, df['pred_win_proba'].values, threshold=thr)
    r['threshold'] = thr
    results.append(r)
thr_df = pd.DataFrame(results)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
thr_df.plot(x='threshold', y='roi', ax=axes[0], title='閾値 vs 回収率 (%)', marker='o')
axes[0].axhline(100, color='red', linestyle='--', label='損益分岐')
axes[0].legend()
thr_df.plot(x='threshold', y='n_bets', ax=axes[1], title='閾値 vs 購入点数', marker='o', color='orange')
thr_df.plot(x='threshold', y='hit_rate', ax=axes[2], title='閾値 vs 的中率 (%)', marker='o', color='green')
plt.tight_layout()
plt.show()
print(thr_df[['threshold','n_bets','hit_rate','roi']].to_string(index=False))

## 2. 予測確率 × オッズ帯域でのマトリクス分析

In [ ]:
df['pred_bin'] = pd.cut(df['pred_win_proba'], bins=[0, 0.1, 0.2, 0.3, 0.5, 1.0],
                         labels=['<10%', '10-20%', '20-30%', '30-50%', '>50%'])
df['odds_bin'] = pd.cut(df['odds_win'], bins=[1, 3, 6, 10, 20, 999],
                         labels=['1-3倍', '3-6倍', '6-10倍', '10-20倍', '20倍超'])

matrix = df.groupby(['pred_bin', 'odds_bin'], observed=True).apply(
    lambda g: roi_by_bet_condition(g, pd.Series(True, index=g.index))['roi']
).unstack()

plt.figure(figsize=(10, 6))
sns.heatmap(matrix, annot=True, fmt='.0f', cmap='RdYlGn', center=100,
            linewidths=0.5, cbar_kws={'label': '回収率 (%)'})
plt.title('予測確率 × オッズ帯域 回収率マトリクス')
plt.xlabel('オッズ帯域')
plt.ylabel('予測勝利確率')
plt.tight_layout()
plt.show()

## 3. ケリー基準によるベットサイズ分析

In [ ]:
# 0.25ケリーで正の期待値がある馬のみ購入
kelly = kelly_bet_sizes(df['pred_win_proba'].values, df['odds_win'].values, fraction=0.25)
df['kelly'] = kelly

# ケリーが正の馬のみを対象に、重み付き回収率シミュレーション
kelly_mask = df['kelly'] > 0
print(f'ケリー正例数: {kelly_mask.sum()} / {len(df)}')

# 単純に正ケリーの馬をフラットベット購入した場合
r = roi_by_bet_condition(df, kelly_mask)
print('\n[正ケリー馬 フラットベット]')
for k, v in r.items():
    print(f'  {k}: {v}')

In [ ]:
# ケリー値の分布
df[df['kelly'] > 0]['kelly'].hist(bins=50, edgecolor='black')
plt.title('正ケリー馬のケリー値分布')
plt.xlabel('Kelly fraction')
plt.ylabel('頭数')
plt.tight_layout()
plt.show()

## 4. 時系列での累積損益シミュレーション

In [ ]:
STRATEGIES = {
    '全馬フラット': pd.Series(True, index=df.index),
    '1番人気のみ': df['popularity'] == 1,
    'モデル予測>20%': df['pred_win_proba'] > 0.20,
    'モデル予測>30%': df['pred_win_proba'] > 0.30,
    '正ケリー': kelly_mask,
}

plt.figure(figsize=(14, 6))
for label, mask in STRATEGIES.items():
    sub = df[mask].copy()
    sub['net'] = sub['payout_win'] - 100  # 100円賭けの損益
    sub = sub.sort_values('race_date')
    cum = sub['net'].cumsum().values
    plt.plot(cum, label=label, alpha=0.85)

plt.axhline(0, color='black', linestyle='--', linewidth=0.8)
plt.title('戦略別 累積損益シミュレーション（100円/点）')
plt.xlabel('購入回数')
plt.ylabel('累積損益（円）')
plt.legend()
plt.tight_layout()
plt.show()

print('\n[戦略別サマリー]')
for label, mask in STRATEGIES.items():
    r = roi_by_bet_condition(df, mask)
    print(f"{label:20s}: n={r['n_bets']:5d}, 的中率={r['hit_rate']:5.1f}%, ROI={r['roi']:6.1f}%")

## 5. ファクター別 ROI（連続変数）

In [ ]:
factors = ['log_odds', 'jockey_te_place', 'trainer_te_place', 'weight_diff']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, col in zip(axes.flat, factors):
    roi_df = roi_by_factor(df, col, bins=8)
    roi_df['roi'].plot(kind='bar', ax=ax, title=f'{col} 別 回収率 (%)')
    ax.axhline(100, color='red', linestyle='--', linewidth=0.8)
    ax.set_xlabel(col)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 6. 回収率向上のための洞察まとめ

In [ ]:
print('=' * 60)
print('回収率向上ファクター分析 サマリー')
print('=' * 60)

# 各戦略のROIを整理
summary = []
for label, mask in STRATEGIES.items():
    r = roi_by_bet_condition(df, mask)
    summary.append({'戦略': label, '点数': r['n_bets'], '的中率(%)': r['hit_rate'], 'ROI(%)': r['roi']})

summary_df = pd.DataFrame(summary).sort_values('ROI(%)', ascending=False)
print(summary_df.to_string(index=False))
print()
print('推奨ファクター優先順位:')
print('  1. オッズ（市場効率の逆張り機会を探す）')
print('  2. モデル予測確率（オッズと乖離した馬が候補）')
print('  3. 騎手の実力（Target Encoding: jockey_te_place）')
print('  4. 調教師の実力（Target Encoding: trainer_te_place）')
print('  5. 距離適性（馬の出走距離パターン）')
print('  6. 馬場状態（稍重以上での変化）')
print('  7. 馬体重変化（増減の方向性より絶対値）')